# Pet segmentation: training v2 (foreground distribution)

This is v2 of `Pet_Segmentation_Training.ipynb`. It applies the **foreground distribution** findings:

| Finding | Change in v2 |
|---|---|
| Mean pet share is about 30%, median 27%, right-skewed (mostly 10–50%, a few up to 95%) | The exact distribution is computed and plotted on the training masks |
| Pets are medium-to-large objects, so this isn't a small-object problem (5% of a 256×256 image is about 3,000 px) | Input stays 256×256 |
| Pixel split is roughly 58% background, 30% pet, 12% boundary | The exact class shares are computed and printed before training |
| About 2:1 background to pet is moderate imbalance | **No class weights** by default: plain CE + Dice (`class_weighting` can switch back to v1's `1/sqrt(freq)`) |
| Images where the pet covers under 10% will likely score worst | Test IoU is also **grouped by pet size** (<10%, 10–30%, 30–50%, >50%) |

Checkpoints go to `checkpoints_v2/`, so the v1 model in `checkpoints/` isn't touched.

---

This notebook trains a **U-Net with an ImageNet-pretrained ResNet-34 encoder** on the Oxford-IIIT Pet trimaps, on a **GPU**.
It follows on from `EDA.ipynb`, and the main choices come from what the EDA found:

| EDA finding | Choice made here |
|---|---|
| Every image has a matching trimap of the same size, with values only {1, 2, 3} | Masks are loaded as-is and remapped to 0 background, 1 pet, 2 boundary |
| Background is most of the pixels, but only about 2:1 against the pet | Cross-entropy **plus** Dice loss, without class weights, and scale-jitter crops |
| The boundary band (class 3) is a wide region of uncertain labels | `boundary_mode`: train it as its own class, or leave it out of the loss |
| A few masks are empty, and some images are near-duplicates across splits | Uses the cleaned lists in `clean_splits/` from EDA section 8 when they exist |
| Pets are centre-biased and often touch the image border | Random resized crops + flips so the model can't rely on position |
| Cats and dogs and the 37 breeds differ in size and edge quality | Test IoU is broken down by species, by breed and by pet size |

**Runtime:** *Runtime ▸ Change runtime type ▸ T4 GPU* (or better), then *Run all*.

**If Colab disconnects:** checkpoints are saved to Drive after every epoch (`checkpoints_v2/` in the project folder).
Choose *Run all* again and training resumes from the last finished epoch.

## 1. Setup

In [ ]:
import os, sys, random, shutil, tarfile, time, urllib.request
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import tv_tensors
from torchvision.transforms import v2
from tqdm.auto import tqdm

IN_COLAB = "google.colab" in sys.modules
print("torch", torch.__version__, "| torchvision", torchvision.__version__)

In [ ]:
CONFIG = {
    "img_size": 256,          # must be divisible by 32
    "batch_size": 16,
    "epochs": 40,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "boundary_mode": "class", # "class": boundary is its own class | "ignore": boundary pixels are left out of the loss
    "dice_weight": 1.0,
    "class_weighting": "none",  # "none": plain CE + Dice | "sqrt_inv": v1's 1/sqrt(pixel frequency) weights
    "patience": 7,           # early stopping on validation mIoU
    "num_workers": 2,
    "seed": 42,
    "pretrained": True,
}
REQUIRE_GPU = True

DATA_DIR = Path("/content/dataset")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/upgaming/Animal Classification & Segmentation")
# on Colab, checkpoints live on Drive so a disconnect doesn't lose the run
CKPT_DIR = DRIVE_PROJECT_DIR / "checkpoints_v2" if IN_COLAB else Path("/content/checkpoints_v2")
SIZE_BINS = [0, 0.10, 0.30, 0.50, 1.0]   # pet share of the image, for the size-grouped evaluation
SIZE_LABELS = ["<10%", "10–30%", "30–50%", ">50%"]
RESUME = True   # continue from CKPT_DIR/last.pt if it exists and was made with the same CONFIG
assert CONFIG["img_size"] % 32 == 0

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))
elif REQUIRE_GPU:
    raise RuntimeError("No GPU found. In Colab choose Runtime ▸ Change runtime type ▸ T4 GPU, then run all again.")
else:
    device = torch.device("cpu")
    print("Running on CPU")
torch.backends.cudnn.benchmark = True


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(CONFIG["seed"])

## 2. Data

Looks for the dataset in `/content/dataset`. If it isn't there, it copies the archives from the Drive `Data` folder when they exist,
and otherwise downloads the official Oxford-IIIT Pet archives.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

IMAGES_DIR = DATA_DIR / "images"
ANN_DIR = DATA_DIR / "annotations"
TRIMAPS_DIR = ANN_DIR / "trimaps"
ARCHIVES = {
    "images.tar.gz": "https://thor.robots.ox.ac.uk/datasets/pets/images.tar.gz",
    "annotations.tar.gz": "https://thor.robots.ox.ac.uk/datasets/pets/annotations.tar.gz",
}


def dataset_ready():
    return IMAGES_DIR.is_dir() and TRIMAPS_DIR.is_dir() and (ANN_DIR / "trainval.txt").exists()


def extract(archive):
    print("extracting", archive.name)
    with tarfile.open(archive) as t:
        try:
            t.extractall(DATA_DIR, filter="data")
        except TypeError:  # Python without tarfile filters
            t.extractall(DATA_DIR)


if not dataset_ready():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    drive_data = DRIVE_PROJECT_DIR / "Data"
    for name, url in ARCHIVES.items():
        local = DATA_DIR / name
        on_drive = drive_data / name
        if on_drive.exists():
            print("copying", on_drive)
            shutil.copy(on_drive, local)
        else:
            print("downloading", url)
            urllib.request.urlretrieve(url, local)
        extract(local)
        local.unlink()

assert dataset_ready(), f"dataset not found in {DATA_DIR}"
print("images:", sum(1 for _ in IMAGES_DIR.glob("*.jpg")), "| trimaps:", sum(1 for _ in TRIMAPS_DIR.glob("*.png")))

### Splits
Uses the cleaned lists from EDA section 8 (`clean_splits/train|val|test.txt`) when they exist. Otherwise it rebuilds the
EDA section 7 split: the official `trainval`/`test` lists, an 80/20 breed-stratified train/val split (seed 42), with empty masks dropped.

In [ ]:
CLEAN_DIR = DATA_DIR / "clean_splits"


def read_list(path):
    return [l.split()[0] for l in path.read_text().splitlines() if l.strip() and not l.startswith("#")]


def breed_of(name):
    return name.rsplit("_", 1)[0]


def species_of(name):
    return "cat" if name[0].isupper() else "dog"


def files_exist(name):
    return (not name.startswith("._") and (IMAGES_DIR / f"{name}.jpg").exists()
            and (TRIMAPS_DIR / f"{name}.png").exists())


if all((CLEAN_DIR / f"{s}.txt").exists() for s in ("train", "val", "test")):
    split_source = "clean_splits (EDA section 8)"
    splits = {s: [n for n in read_list(CLEAN_DIR / f"{s}.txt") if files_exist(n)] for s in ("train", "val", "test")}
else:
    from sklearn.model_selection import train_test_split
    split_source = "official lists + 80/20 stratified split (EDA section 7)"
    trainval = [n for n in read_list(ANN_DIR / "trainval.txt") if files_exist(n)]
    test = [n for n in read_list(ANN_DIR / "test.txt") if files_exist(n)]
    train, val = train_test_split(trainval, test_size=0.20, random_state=42, stratify=[breed_of(n) for n in trainval])

    def has_pet(name):
        return (np.array(Image.open(TRIMAPS_DIR / f"{name}.png")) == 1).any()

    splits = {s: [n for n in tqdm(v, desc=f"empty-mask check ({s})") if has_pet(n)]
              for s, v in {"train": train, "val": val, "test": test}.items()}

print("split source:", split_source)
for s, v in splits.items():
    print(f"  {s:5s}: {len(v):5d} images, {len({breed_of(n) for n in v})} breeds")

### Labels
Trimap values are 1 = pet, 2 = background, 3 = boundary. They are remapped to 0 = background, 1 = pet, and either 2 = boundary
(`boundary_mode="class"`) or 255 = ignored (`boundary_mode="ignore"`).

In [ ]:
IGNORE = 255
if CONFIG["boundary_mode"] == "class":
    CLASS_NAMES = ["background", "pet", "boundary"]
    TRIMAP_TO_LABEL = {1: 1, 2: 0, 3: 2}
elif CONFIG["boundary_mode"] == "ignore":
    CLASS_NAMES = ["background", "pet"]
    TRIMAP_TO_LABEL = {1: 1, 2: 0, 3: IGNORE}
else:
    raise ValueError(CONFIG["boundary_mode"])
NUM_CLASSES = len(CLASS_NAMES)

LUT = np.full(256, IGNORE, dtype=np.uint8)
for k, v in TRIMAP_TO_LABEL.items():
    LUT[k] = v


def load_label(name):
    return LUT[np.array(Image.open(TRIMAPS_DIR / f"{name}.png"))]


def load_rgb(name):
    return np.array(Image.open(IMAGES_DIR / f"{name}.jpg").convert("RGB"))

### Dataset and augmentation

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
S = CONFIG["img_size"]

# torchvision v2 transforms apply the same geometry to image and mask (nearest-neighbour for masks);
# colour jitter and normalisation only touch the image.
train_tf = v2.Compose([
    v2.RandomResizedCrop((S, S), scale=(0.5, 1.0), ratio=(0.75, 1.33), antialias=True),
    v2.RandomHorizontalFlip(),
    v2.RandomApply([v2.ColorJitter(0.3, 0.3, 0.3, 0.05)], p=0.8),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])
eval_tf = v2.Compose([
    v2.Resize((S, S), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])


class PetSegDataset(Dataset):
    def __init__(self, names, transform):
        self.names, self.transform = list(names), transform

    def __len__(self):
        return len(self.names)

    def __getitem__(self, i):
        name = self.names[i]
        img = tv_tensors.Image(torch.from_numpy(load_rgb(name)).permute(2, 0, 1))
        mask = tv_tensors.Mask(torch.from_numpy(load_label(name)))
        img, mask = self.transform(img, mask)
        return img, mask.long(), i


datasets = {
    "train": PetSegDataset(splits["train"], train_tf),
    "val": PetSegDataset(splits["val"], eval_tf),
    "test": PetSegDataset(splits["test"], eval_tf),
}
loader_kw = dict(batch_size=CONFIG["batch_size"], num_workers=CONFIG["num_workers"],
                 pin_memory=device.type == "cuda", persistent_workers=CONFIG["num_workers"] > 0)
loaders = {
    "train": DataLoader(datasets["train"], shuffle=True, drop_last=True, **loader_kw),
    "val": DataLoader(datasets["val"], shuffle=False, **loader_kw),
    "test": DataLoader(datasets["test"], shuffle=False, **loader_kw),
}

In [ ]:
def denormalize(x):
    return (x.permute(1, 2, 0).cpu().numpy() * STD + MEAN).clip(0, 1)


def label_to_rgb(m):
    palette = np.array([[40, 40, 40], [228, 26, 28], [255, 217, 47]], dtype=np.uint8)
    m = np.asarray(m)
    out = np.full((*m.shape, 3), 255, dtype=np.uint8)  # white = ignored
    valid = m != IGNORE
    out[valid] = palette[m[valid]]
    return out


imgs, masks, _ = next(iter(loaders["train"]))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for k in range(4):
    axes[0, k].imshow(denormalize(imgs[k]))
    axes[1, k].imshow(label_to_rgb(masks[k]))
    for ax in axes[:, k]:
        ax.axis("off")
fig.suptitle("Augmented training samples (dark = background, red = pet, yellow = boundary)")
plt.tight_layout()
plt.show()

### Foreground distribution and class balance
Exact numbers on the training masks at full resolution. The trimap is read directly, so the shares are the same in both
`boundary_mode`s: **pet share** is pet + boundary (the whole animal) as a fraction of the image.

In [ ]:
trimap_px = np.zeros(4)   # index 1 pet, 2 background, 3 boundary
pet_shares = []
for name in tqdm(splits["train"], desc="pixel counts"):
    t = np.array(Image.open(TRIMAPS_DIR / f"{name}.png"))
    c = np.bincount(t.ravel(), minlength=4)[:4]
    trimap_px += c
    pet_shares.append((c[1] + c[3]) / t.size)
pet_shares = pd.Series(pet_shares, index=splits["train"], name="pet_share")

share = trimap_px / trimap_px[1:].sum()
display(pd.DataFrame({"pixel_share": [share[2], share[1], share[3]]},
                     index=["background", "pet", "boundary"]).round(4))
print(f"background : pet = {share[2] / share[1]:.2f} : 1   (background : pet+boundary = {share[2] / (share[1] + share[3]):.2f} : 1)")
print(f"pet share: mean {pet_shares.mean():.1%}, median {pet_shares.median():.1%}, "
      f"10–50%: {pet_shares.between(0.10, 0.50).mean():.1%} of images, max {pet_shares.max():.1%}")
print(f"at {S}x{S} input, a pet covering 5% of the image is about {0.05 * S * S:,.0f} px")

size_bin_counts = pd.cut(pet_shares, SIZE_BINS, labels=SIZE_LABELS, include_lowest=True).value_counts().reindex(SIZE_LABELS)
display(size_bin_counts.to_frame("train images"))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(pet_shares, bins=50, color="#4c72b0")
for v, lbl, ls in [(pet_shares.mean(), "mean", "--"), (pet_shares.median(), "median", "-")]:
    ax.axvline(v, color="k", linestyle=ls, label=f"{lbl} {v:.1%}")
for b in SIZE_BINS[1:-1]:
    ax.axvline(b, color="#c44e52", alpha=0.4)
ax.set_xlabel("pet share of the image (pet + boundary)")
ax.set_ylabel("training images")
ax.set_title("Foreground distribution (red lines: size-bin edges used in evaluation)")
ax.legend()
plt.tight_layout()
plt.show()

### Class weights
Background vs pet is only about 2:1, so by default the loss is plain cross-entropy + Dice with **no class weights**
(Dice already balances the classes). Set `class_weighting="sqrt_inv"` to use v1's `1/sqrt(frequency)` weights instead.

In [ ]:
counts = np.zeros(NUM_CLASSES)
for k, v in TRIMAP_TO_LABEL.items():
    if v != IGNORE:
        counts[v] += trimap_px[k]
freq = counts / counts.sum()
if CONFIG["class_weighting"] == "sqrt_inv":
    class_weights = 1 / np.sqrt(freq)
    class_weights /= class_weights.mean()
elif CONFIG["class_weighting"] == "none":
    class_weights = np.ones(NUM_CLASSES)
else:
    raise ValueError(CONFIG["class_weighting"])
display(pd.DataFrame({"pixel_share": freq.round(4), "weight": class_weights.round(3)}, index=CLASS_NAMES))

## 3. Model: U-Net with a ResNet-34 encoder

In [ ]:
def conv_bn_relu(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))


class DecoderBlock(nn.Module):
    def __init__(self, c_in, c_skip, c_out):
        super().__init__()
        self.block = nn.Sequential(conv_bn_relu(c_in + c_skip, c_out), conv_bn_relu(c_out, c_out))

    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        return self.block(x)


class UNetResNet34(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        r = torchvision.models.resnet34(weights=torchvision.models.ResNet34_Weights.DEFAULT if pretrained else None)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)          # 1/2,  64 ch
        self.pool = r.maxpool
        self.enc1, self.enc2, self.enc3, self.enc4 = r.layer1, r.layer2, r.layer3, r.layer4  # 1/4 .. 1/32
        self.dec4 = DecoderBlock(512, 256, 256)
        self.dec3 = DecoderBlock(256, 128, 128)
        self.dec2 = DecoderBlock(128, 64, 64)
        self.dec1 = DecoderBlock(64, 64, 32)
        self.dec0 = DecoderBlock(32, 0, 16)
        self.head = nn.Conv2d(16, num_classes, 1)

    def forward(self, x):
        s0 = self.stem(x)
        e1 = self.enc1(self.pool(s0))
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        d = self.dec4(e4, e3)
        d = self.dec3(d, e2)
        d = self.dec2(d, e1)
        d = self.dec1(d, s0)
        return self.head(self.dec0(d))


model = UNetResNet34(NUM_CLASSES, CONFIG["pretrained"]).to(device).to(memory_format=torch.channels_last)
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M")

## 4. Loss and metrics

In [ ]:
class CEDiceLoss(nn.Module):
    """Class-weighted cross-entropy + soft Dice; pixels labelled IGNORE count for neither."""

    def __init__(self, weight, dice_weight=1.0):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=weight, ignore_index=IGNORE)
        self.dice_weight = dice_weight

    def forward(self, logits, target):
        ce = self.ce(logits, target)
        valid = (target != IGNORE).unsqueeze(1)
        prob = logits.float().softmax(dim=1) * valid
        onehot = F.one_hot(torch.where(target == IGNORE, 0, target), logits.shape[1]).permute(0, 3, 1, 2) * valid
        inter = (prob * onehot).sum(dim=(0, 2, 3))
        denom = prob.sum(dim=(0, 2, 3)) + onehot.sum(dim=(0, 2, 3))
        dice = 1 - ((2 * inter + 1) / (denom + 1)).mean()
        return ce + self.dice_weight * dice


def confusion(pred, target, n=NUM_CLASSES):
    k = target != IGNORE
    return torch.bincount(target[k] * n + pred[k], minlength=n * n).reshape(n, n)


def scores(cm):
    cm = cm.double()
    tp = cm.diag()
    iou = tp / (cm.sum(0) + cm.sum(1) - tp).clamp(min=1)
    return {"miou": iou.mean().item(), "pixel_acc": (tp.sum() / cm.sum().clamp(min=1)).item(),
            **{f"iou_{c}": iou[i].item() for i, c in enumerate(CLASS_NAMES)}}


def pet_iou(pred, target):
    """Per-image IoU of the whole animal (pet + boundary) vs background, over non-ignored pixels."""
    valid = target != IGNORE
    p, t = (pred != 0) & valid, (target != 0) & valid
    inter = (p & t).flatten(1).sum(1).float()
    union = (p | t).flatten(1).sum(1).float()
    return torch.where(union > 0, inter / union.clamp(min=1), torch.ones_like(union))


criterion = CEDiceLoss(torch.tensor(class_weights, dtype=torch.float32, device=device), CONFIG["dice_weight"])

## 5. Training

In [ ]:
use_amp = device.type == "cuda"
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=CONFIG["lr"], pct_start=0.1,
                                                total_steps=CONFIG["epochs"] * len(loaders["train"]))
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)


def run_epoch(loader, train):
    model.train(train)
    total_loss, n_seen = 0.0, 0
    cm = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long, device=device)
    pet_ious = []
    with torch.set_grad_enabled(train):
        for x, y, _ in tqdm(loader, leave=False, desc="train" if train else "eval"):
            x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
            y = y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x)
                loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
            pred = logits.argmax(1)
            cm += confusion(pred, y)
            pet_ious.append(pet_iou(pred, y))
            total_loss += loss.item() * len(x)
            n_seen += len(x)
    return {"loss": total_loss / n_seen, **scores(cm), "pet_iou": torch.cat(pet_ious).mean().item()}


CKPT_DIR.mkdir(parents=True, exist_ok=True)
best_path, last_path = CKPT_DIR / "best.pt", CKPT_DIR / "last.pt"
history, best_miou, bad_epochs, start_epoch = [], -1.0, 0, 1

if RESUME and last_path.exists():
    last = torch.load(last_path, map_location=device, weights_only=False)
    if last["config"] == CONFIG:
        model.load_state_dict(last["model"])
        optimizer.load_state_dict(last["optimizer"])
        scheduler.load_state_dict(last["scheduler"])
        scaler.load_state_dict(last["scaler"])
        history, best_miou, bad_epochs = last["history"], last["best_miou"], last["bad_epochs"]
        start_epoch = last["epoch"] + 1
        print(f"resuming after epoch {last['epoch']} (best val mIoU so far {best_miou:.4f})")
    else:
        print("last.pt was made with a different CONFIG, starting from scratch")
    del last

stopped_early = bool(history) and bad_epochs >= CONFIG["patience"]
for epoch in range(start_epoch, CONFIG["epochs"] + 1):
    if stopped_early:
        break
    t0 = time.time()
    tr = run_epoch(loaders["train"], train=True)
    va = run_epoch(loaders["val"], train=False)
    history.append({"epoch": epoch, "lr": scheduler.get_last_lr()[0],
                    **{f"train_{k}": v for k, v in tr.items()}, **{f"val_{k}": v for k, v in va.items()}})
    improved = va["miou"] > best_miou
    if improved:
        best_miou, bad_epochs = va["miou"], 0
        torch.save({"model": model.state_dict(), "config": CONFIG, "class_names": CLASS_NAMES, "epoch": epoch}, best_path)
    else:
        bad_epochs += 1
    print(f"epoch {epoch:3d} | {time.time() - t0:5.0f}s | train loss {tr['loss']:.4f} mIoU {tr['miou']:.4f} | "
          f"val loss {va['loss']:.4f} mIoU {va['miou']:.4f} pet IoU {va['pet_iou']:.4f}" + ("  *best*" if improved else ""))
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(), "history": history, "best_miou": best_miou, "bad_epochs": bad_epochs,
                "epoch": epoch, "config": CONFIG}, last_path)
    if bad_epochs >= CONFIG["patience"]:
        print(f"early stopping: no val mIoU improvement for {CONFIG['patience']} epochs")
        stopped_early = True

hist = pd.DataFrame(history)
hist.to_csv(CKPT_DIR / "history.csv", index=False)
print(f"best val mIoU {best_miou:.4f} -> {best_path}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].plot(hist.epoch, hist.train_loss, label="train")
axes[0].plot(hist.epoch, hist.val_loss, label="val")
axes[0].set_title("loss")
axes[1].plot(hist.epoch, hist.train_miou, label="train")
axes[1].plot(hist.epoch, hist.val_miou, label="val")
axes[1].set_title("mIoU")
for c in CLASS_NAMES:
    axes[2].plot(hist.epoch, hist[f"val_iou_{c}"], label=c)
axes[2].plot(hist.epoch, hist.val_pet_iou, "k--", label="pet (whole animal)")
axes[2].set_title("validation IoU per class")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Test-set evaluation
Uses the best checkpoint. Metrics are computed at the training resolution (`img_size`). **Pet IoU** treats pet + boundary as
the animal vs background. The boundary band's own IoU is reported separately, because its labels are uncertain by design.

In [ ]:
ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt["model"])
model.eval()

cm = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long, device=device)
rows = []
with torch.no_grad():
    for x, y, idx in tqdm(loaders["test"], desc="test"):
        x = x.to(device).to(memory_format=torch.channels_last)
        y = y.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            pred = model(x).argmax(1)
        cm += confusion(pred, y)
        shares = (y != 0).flatten(1).float().mean(1)   # pet + boundary (IGNORE counts as boundary)
        for i, piou, sh in zip(idx.tolist(), pet_iou(pred, y).tolist(), shares.tolist()):
            name = datasets["test"].names[i]
            rows.append({"filename": name, "breed": breed_of(name), "species": species_of(name),
                         "pet_share": sh, "pet_iou": piou})

test_scores = scores(cm)
per_image = pd.DataFrame(rows)
test_scores["pet_iou"] = per_image.pet_iou.mean()
print(f"best epoch: {ckpt['epoch']}")
display(pd.Series(test_scores, name="test").round(4).to_frame())

In [ ]:
display(per_image.groupby("species").pet_iou.agg(["mean", "median", "count"]).round(4))

breed_iou = per_image.groupby(["species", "breed"]).pet_iou.mean().reset_index().sort_values("pet_iou")
fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(breed_iou.breed, breed_iou.pet_iou, color=breed_iou.species.map({"cat": "#1f77b4", "dog": "#ff7f0e"}))
ax.set_ylim(max(0, breed_iou.pet_iou.min() - 0.05), 1)
ax.set_ylabel("mean pet IoU")
ax.set_title("Test pet IoU by breed (blue = cat, orange = dog)")
ax.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()
print("hardest breeds:", ", ".join(breed_iou.head(5).breed))

### IoU by pet size
Test images grouped by how much of the image the pet covers. Small pets (<10%) are expected to score worst.

In [ ]:
per_image["size_bin"] = pd.cut(per_image.pet_share, SIZE_BINS, labels=SIZE_LABELS, include_lowest=True)
by_size = (per_image.groupby("size_bin", observed=False)
           .agg(images=("pet_iou", "size"), mean_pet_share=("pet_share", "mean"),
                mean_pet_iou=("pet_iou", "mean"), median_pet_iou=("pet_iou", "median"))
           .reindex(SIZE_LABELS))
display(by_size.round(4))

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(by_size.index.astype(str), by_size.mean_pet_iou, color="#4c72b0")
for b, n in zip(bars, by_size.images):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f"n={n}", ha="center", va="bottom")
ax.set_ylim(max(0, by_size.mean_pet_iou.min() - 0.1), 1)
ax.set_xlabel("pet share of the image")
ax.set_ylabel("mean pet IoU")
ax.set_title("Test pet IoU by pet size")
plt.tight_layout()
plt.show()

In [ ]:
def show_predictions(names, title):
    fig, axes = plt.subplots(len(names), 3, figsize=(11, 3.4 * len(names)), squeeze=False)
    ds = PetSegDataset(names, eval_tf)
    with torch.no_grad():
        for row, (x, y, i) in zip(axes, ds):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                pred = model(x[None].to(device).to(memory_format=torch.channels_last)).argmax(1)[0].cpu()
            piou = pet_iou(pred[None], y[None]).item()
            row[0].imshow(denormalize(x))
            row[1].imshow(label_to_rgb(y))
            row[2].imshow(label_to_rgb(pred))
            row[0].set_title(ds.names[i])
            row[1].set_title("ground truth")
            row[2].set_title(f"prediction (pet IoU {piou:.2f})")
            for ax in row:
                ax.axis("off")
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


ranked = per_image.sort_values("pet_iou")
show_predictions(ranked.head(4).filename.tolist(), "Worst test predictions")
show_predictions(ranked.tail(4).filename.tolist()[::-1], "Best test predictions")

## 7. Save results to Drive

In [ ]:
per_image.to_csv(CKPT_DIR / "test_per_image.csv", index=False)
pd.Series(test_scores).to_csv(CKPT_DIR / "test_scores.csv")

if IN_COLAB and DRIVE_PROJECT_DIR.exists():
    run_dir = DRIVE_PROJECT_DIR / "runs" / datetime.now().strftime("v2_%Y%m%d_%H%M%S")
    run_dir.mkdir(parents=True, exist_ok=True)
    for f in ["best.pt", "history.csv", "test_per_image.csv", "test_scores.csv"]:
        shutil.copy(CKPT_DIR / f, run_dir / f)
    print("saved to", run_dir)
else:
    print("results kept in", CKPT_DIR)